# LidarChange

#### Notebook for processing and visualizing point-cloud-based lidar change detection

Includes tools for reprojection, tiling, ICP alignment, M3C2 change detection, raster visualization, and point profile creation
  
Last Modified September 23, 2026  

Brandon T. Fong and Roman A. DiBiase  
Department of Geosciences, Pennsylvania State University  
contact: brandonfong@psu.edu  

**AI-assisted development:** *This notebook was developed with assistance from Anthropic's Claude Code. Code generated or modified with Claude Code was reviewed and tested by the authors, who are responsible for the final product.*

## Tips for running the notebook
#### Input data:
- Organize project in its own folder, such as /Inputs/SampleData/
- Within this folder, dump raw .las/.laz files into one folder for each epoch
- Generate an AOI for your analysis extent. Quickest is to use https://geojson.io/ and export as shapefile
- It may be helpful to "chunk" larger areas (>100 km<sup>2</sup>) into multiple AOIs to keep output merged raster size manageable
- For reference, ~10 km<sup>2</sup> AOI should take on the order of 10-20 minutes on laptop
#### Coordinate systems
- An internet connection is needed for datum transformations
- If vertical CRS is unknown, at least know if units are meters or feet
#### Other parameters
- Default values are a good starting point for typical airborne lidar data in mountainous landscapes (~1-10 pts/m2 ground point density).
#### Output data:
- Output folder is generated for each run with a timestamp, including
    - Projected and tiled ground-classified point clouds (for both "before" and "after")
    - Aligned "after" point clouds with M3C2 distance scalar field
    - DEM, slope, and hillshade raster files for "before" and "after"
    - Gridded M3C2 change detection raster
    - Visualizations generated in this notebook (maps and cross sections)
- If changing parameters and re-running, edit Cell 5 (parameters) and then run Cell 6 to generate a new output folder. Preprocessing (Cell 4) does not need to be re-run
#### Preprocessed point clouds:
- Cell 4's ground-classified, clipped, reprojected clouds are cached in a `Preprocessed` folder at the project root, outside the timestamped output folders
- They are shared by every run and rebuilt only when the input folders, the AOI, or the output CRS change
- Delete that folder any time to reclaim disk space; the next run will rebuild it
#### Point cloud profiles:
- Make sure to change profile title (i.e., "A", "B", etc.) if saving multiple profiles
- Note different approaches to cross-slope tilt correction in Cell 10

## 1. Prepare environment and load libraries

In [ ]:
## Do not edit the contents of this cell
## Can take up to a minute or longer to import CloudComPy

import sys
import shutil
from pathlib import Path

## Define project root (folder containing this notebook) -- inserted onto sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Prepare environment to avoid DLL issues with CloudComPy
from LidarChangeScripts.environment_setup import prepare_environment, define_project_directories, define_preprocess_dir
PDAL_ENV = prepare_environment()

## Project-level cache of preprocessed point clouds, shared by every run
PREPROCESSED_DIR = define_preprocess_dir(PROJECT_ROOT)

from LidarChangeScripts.path_utils import pick_directory, pick_file, prompt_epsg_code, prompt_dataset_crs
from LidarChangeScripts.run_record import write_run_record, warn_if_parameters_changed
from LidarChangeScripts.preprocess_cache import (
    preprocess_fingerprint,
    cache_status,
    cached_file_counts,
    clear_preprocess_cache,
    write_preprocess_fingerprint,
    copy_preprocess_artifacts
)
import LidarChangeScripts
from osgeo import ogr
ogr.UseExceptions()  # Python error handling
print(f"Importing CloudComPy libraries")
import cloudComPy as cc

from LidarChangeScripts.crs_utils import (
    describe_crs,
    crs_linear_unit_m,
    prompt_output_crs,
    reproject_polygon_wkt
)
from LidarChangeScripts.las_metadata import get_dataset_crs, files_overlapping_polygon
from LidarChangeScripts.point_density import estimate_ground_point_density
from LidarChangeScripts.survey_dates import survey_date_range
from LidarChangeScripts.preprocessing import create_pdal_pipeline, preprocess_files
from LidarChangeScripts.tiling import retile_clouds, archive_tiles
from LidarChangeScripts.tile_merging import merge_partial_tiles

from LidarChangeScripts.boundary_utils import (
    create_extent_shapefile,
    shapefile_to_wkt,
    shapefile_crs_info,
    buffer_polygon_wkt,
    polygon_area_m2
)

from LidarChangeScripts.m3c2_utils import create_m3c2_parameter_files

from LidarChangeScripts.m3c2_processing import M3C2_MODES, M3C2_CHANGE_LABELS

from LidarChangeScripts.cloudcompy_utils import process_tile_pairs
print(f"Successfully imported libraries (LidarChangeScripts version {LidarChangeScripts.__version__})")

## 2. Input data folder location definition

In [ ]:
## This cell generates pop-up dialogs for selecting the input data folder locations
## Also confirms coordinate reference system (CRS) for outputs

## Define Inputs directory, where point cloud and analysis extent shapefile live
## Used as the starting folder for the dialogs below.
INPUTS_DIR = PROJECT_ROOT / "Inputs"

DATABEFORE = pick_directory(
    "Select the BEFORE-event LAS/LAZ folder",
    initialdir=INPUTS_DIR
)
DATAAFTER = pick_directory(
    "Select the AFTER-event LAS/LAZ folder",
    initialdir=INPUTS_DIR
)
POLYGON = pick_file(
    "Select the analysis-extent polygon shapefile",
    filetypes=[("Shapefile", "*.shp")],
    initialdir=INPUTS_DIR
)

before_files = sorted(list(DATABEFORE.glob("*.las")) + list(DATABEFORE.glob("*.laz")))
after_files = sorted(list(DATAAFTER.glob("*.las")) + list(DATAAFTER.glob("*.laz")))

if not before_files:
    raise RuntimeError(f"No LAS/LAZ files found in {DATABEFORE}")
if not after_files:
    raise RuntimeError(f"No LAS/LAZ files found in {DATAAFTER}")

print(f"BEFORE folder: {DATABEFORE} ({len(before_files)} files)")
print(f"AFTER folder: {DATAAFTER} ({len(after_files)} files)")
print(f"AOI shapefile: {POLYGON}")

## Detect each input's CRS, asking only for what can't be read from the files
print("\n--- Coordinate reference systems ---")

aoi_srs_wkt, aoi_epsg, _ = shapefile_crs_info(POLYGON)
aoi_crs = aoi_epsg if aoi_epsg else None

if not aoi_crs:
    aoi_crs = prompt_epsg_code(
        f"No EPSG code found in the AOI shapefile's CRS ({POLYGON.name}).\n"
        f"Enter its EPSG code (e.g. EPSG:26911):"
    )
print(f"AOI shapefile CRS: {aoi_crs} ({describe_crs(aoi_crs)})")

before_crs, before_horizontal_crs = get_dataset_crs(before_files, "BEFORE", env=PDAL_ENV)
if not before_crs:
    before_crs = prompt_dataset_crs("BEFORE", DATABEFORE.name, before_horizontal_crs, example="EPSG:2229+6360")
print(f"BEFORE dataset CRS: {before_crs} ({describe_crs(before_crs)})")

after_crs, after_horizontal_crs = get_dataset_crs(after_files, "AFTER", env=PDAL_ENV)
if not after_crs:
    after_crs = prompt_dataset_crs("AFTER", DATAAFTER.name, after_horizontal_crs, example="EPSG:6340+5703")
print(f"AFTER dataset CRS: {after_crs} ({describe_crs(after_crs)})")

## Define output CRS using dialog pre-filled with the AFTER dataset CRS
OUTPUT_CRS = prompt_output_crs(default_crs=after_crs)
print(f"\nOUTPUT CRS: {OUTPUT_CRS} ({describe_crs(OUTPUT_CRS)})")


## 3. Input data diagnostics

In [ ]:
## Calculates the ground-classified point density of each dataset within the AOI
## Calendar dates each survey was flown are read from the GPS time scalar field

## AOI polygon, read straight from the shapefile
aoi_wkt = shapefile_to_wkt(POLYGON)

## Calculate area of AOI in square meters (true ground area, whatever the shapefile's CRS)
aoi_area_m2 = polygon_area_m2(aoi_wkt, aoi_srs_wkt)

print(f"\nAOI area: {aoi_area_m2:,.1f} m^2")

## Only files whose extent overlaps the AOI are counted below
print("\n--- Files overlapping the AOI ---")
before_aoi_files = files_overlapping_polygon(before_files, aoi_wkt, aoi_srs_wkt, before_crs, "BEFORE", env=PDAL_ENV)
after_aoi_files = files_overlapping_polygon(after_files, aoi_wkt, aoi_srs_wkt, after_crs, "AFTER", env=PDAL_ENV)

## Estimate "before" ground point density
print("\n--- BEFORE ground-point density ---")
before_points, before_density, before_gps_range = estimate_ground_point_density(
    files=before_aoi_files,
    aoi_wkt=aoi_wkt,
    aoi_srs_wkt=aoi_srs_wkt,
    aoi_area_m2=aoi_area_m2,
    dataset_crs=before_crs,
    env=PDAL_ENV
)
print(f"BEFORE ground point density: {before_density:.2f} pts/m^2 ({before_points:,} points in AOI)")

## Estimate "after" ground point density
print("\n--- AFTER ground-point density ---")
after_points, after_density, after_gps_range = estimate_ground_point_density(
    files=after_aoi_files,
    aoi_wkt=aoi_wkt,
    aoi_srs_wkt=aoi_srs_wkt,
    aoi_area_m2=aoi_area_m2,
    dataset_crs=after_crs,
    env=PDAL_ENV
)
print(f"AFTER ground point density: {after_density:.2f} pts/m^2 ({after_points:,} points in AOI)")

## Dates each survey was flown, from the GPS time scalar field
print("\n--- Survey dates ---")
print(f"BEFORE survey dates: {survey_date_range(before_gps_range, before_aoi_files, env=PDAL_ENV)}")
print(f"AFTER survey dates: {survey_date_range(after_gps_range, after_aoi_files, env=PDAL_ENV)}")

## Stop here if either dataset has no ground points in the AOI
for label, points in (("BEFORE", before_points), ("AFTER", after_points)):
    if points == 0:
        raise RuntimeError(
            f"The {label} dataset has no ground-classified points inside AOI"
        )

## 4. Preprocess raw lidar point clouds (filter, clip, and project)

In [ ]:
## Preprocessing of raw lidar point clouds
## Extracts ground points, clips to AOI, and projects to output CRS
## Results are cached in PREPROCESSED_DIR and shared by every run, so the
## parameters in Cell 5 can be changed without repeating this cell's work
##

## Preprocessing options
AOI_BUFFER = 10 # meters of padding around the AOI, for edge calculations
MAKE_EXTENT_SHAPEFILES = False # Set to True to generate shapefiles of before/after lidar extent
FORCE_PREPROCESS = False # Set to True to rebuild the cache even when it looks current

## AOI polygon, reprojected from the shapefile's own CRS into OUTPUT_CRS
crop_polygon = reproject_polygon_wkt(aoi_wkt, aoi_srs_wkt, OUTPUT_CRS)

## Padded AOI for edge calculations
crop_polygon_padded = buffer_polygon_wkt(crop_polygon, AOI_BUFFER)

## Only process files whose extent overlaps the padded AOI
before_run_files = files_overlapping_polygon(before_files, crop_polygon_padded, OUTPUT_CRS, before_crs, "BEFORE", env=PDAL_ENV)
after_run_files = files_overlapping_polygon(after_files, crop_polygon_padded, OUTPUT_CRS, after_crs, "AFTER", env=PDAL_ENV)

## Everything the preprocessed clouds depend on, checked against the cache
preprocess_config = preprocess_fingerprint(
    before_dir=DATABEFORE,
    after_dir=DATAAFTER,
    before_files=before_run_files,
    after_files=after_run_files,
    before_crs=before_crs,
    after_crs=after_crs,
    output_crs=OUTPUT_CRS,
    aoi_path=POLYGON,
    crop_polygon_padded=crop_polygon_padded,
    aoi_buffer=AOI_BUFFER,
    make_extent_shapefiles=MAKE_EXTENT_SHAPEFILES
)

cache_reusable, cache_reasons = cache_status(
    PREPROCESSED_DIR,
    preprocess_config,
    force=FORCE_PREPROCESS
)

if cache_reusable:

    cached_counts = cached_file_counts(PREPROCESSED_DIR)
    print(f"\nUsing cached preprocessing in {PREPROCESSED_DIR}")
    print(f"  {cached_counts['before']} BEFORE file(s), {cached_counts['after']} AFTER file(s)")
    print(f"  Set FORCE_PREPROCESS = True above to rebuild it")

else:

    print("\nPreprocessing raw point clouds:")
    for reason in cache_reasons:
        print(f"  {reason}")

    ## Clear any earlier clouds first
    clear_preprocess_cache(PREPROCESSED_DIR)

    ## Create PDAL processing pipelines for projection, crop to AOI, and filter ground points
    before_pipeline = PREPROCESSED_DIR / "before_pipeline.json"

    create_pdal_pipeline(
        pipeline_path=before_pipeline,
        input_crs=before_crs,
        output_crs=OUTPUT_CRS,
        crop_polygon=crop_polygon_padded,
        classify_ground=True,
        compress_output=False,
        compute_boundary=MAKE_EXTENT_SHAPEFILES
    )

    after_pipeline = PREPROCESSED_DIR / "after_pipeline.json"

    create_pdal_pipeline(
        pipeline_path=after_pipeline,
        input_crs=after_crs,
        output_crs=OUTPUT_CRS,
        crop_polygon=crop_polygon_padded,
        classify_ground=True,
        compress_output=False,
        compute_boundary=MAKE_EXTENT_SHAPEFILES
    )

    ## Run PDAL processing pipeline on "before" las/laz files, saving to the cache
    before_boundaries = preprocess_files(
        files=before_run_files,
        pipeline_file=before_pipeline,
        output_suffix="_processed",
        output_dir=PREPROCESSED_DIR/"before",
        output_extension=".las",
        label="BEFORE",
        env=PDAL_ENV
    )

    ## Same for "after" las/laz files
    after_boundaries = preprocess_files(
        files=after_run_files,
        pipeline_file=after_pipeline,
        output_suffix="_processed",
        output_dir=PREPROCESSED_DIR/"after",
        output_extension=".las",
        label="AFTER",
        env=PDAL_ENV
    )

    ## Save raw-data extent shapefiles into the cache
    if MAKE_EXTENT_SHAPEFILES:
        create_extent_shapefile(
            boundaries=before_boundaries,
            output_shp=PREPROCESSED_DIR/"before_raw_lidar_extent.shp",
            crs=before_crs,
            name="before_raw_lidar_extent"
        )
        create_extent_shapefile(
            boundaries=after_boundaries,
            output_shp=PREPROCESSED_DIR/"after_raw_lidar_extent.shp",
            crs=after_crs,
            name="after_raw_lidar_extent"
        )

    ## Write preprocessing fingerprint
    write_preprocess_fingerprint(PREPROCESSED_DIR, preprocess_config)
    print(f"\nPreprocessed clouds cached in {PREPROCESSED_DIR}")

## 5. Define main parameters

In [ ]:
## Define main parameters, including tiling, ICP, M3C2, and plotting parameters
## Tile sizes, buffers, spacing and M3C2 scales below are all in meters.

## Tiling parameters
TILE_SIZE = 500                 # meters (recommended 500-2000 m)
BUFFER_DISTANCE = 10            # meters (should be larger than NORMAL_SCALE and PROJECTION_SCALE)
PARTIAL_TILE_MIN_PERCENT = 20   # threshold size for tiles to merge as percentage size of largest tile

## Raster parameters (make sure tile size is integer multiple of resolution)
# DEM output resolution (m/pixel)
BEFORE_DEM_RESOLUTION = 1
AFTER_DEM_RESOLUTION = 1

# M3C2 change raster output resolution (m/pixel)
M3C2_RASTER_RESOLUTION = 1

for name, resolution in (
    ("BEFORE_DEM_RESOLUTION", BEFORE_DEM_RESOLUTION),
    ("AFTER_DEM_RESOLUTION", AFTER_DEM_RESOLUTION),
    ("M3C2_RASTER_RESOLUTION", M3C2_RASTER_RESOLUTION),
):
    cells_per_tile = TILE_SIZE / resolution
    if resolution <= 0 or abs(cells_per_tile - round(cells_per_tile)) > 1e-9:
        raise ValueError(
            f"{name} ({resolution} m) must divide evenly into TILE_SIZE "
            f"({TILE_SIZE} m), so raster pixels line up with the tile edges"
        )

## CloudComPy ICP parameters
MIN_SPACING = 0.5             # meters
ICP_SAMPLING_LIMIT = 500_000  # number of points
ICP_MIN_RMS_DECREASE = 1e-5   # meters

## CloudComPy M3C2 parameters
# Diameter of the window used to fit surface normals (meters)
NORMAL_SCALE = 3.0

# Diameter of the cylinder that points are projected into (meters)
PROJECTION_SCALE = 3.0

# Length of the projection cylinder (max change) (meters)
SEARCH_DEPTH = 200

REGISTRATION_ERROR = 0.2       # meters

## M3C2 MODE
#   "Vertical"        -- vertical M3C2
#   "Surface Normal"  -- surface-normal M3C2
M3C2_MODE = "Vertical"

## M3C2 FILTER
# True filters points for significant surface-normal change
# False keeps all points (noisier)
M3C2_FILTER = True

if M3C2_MODE not in M3C2_MODES:
    raise ValueError(
        f"M3C2_MODE must be one of {list(M3C2_MODES)}, got {M3C2_MODE!r}"
    )

if M3C2_FILTER not in (True, False):
    raise ValueError(f"M3C2_FILTER must be True or False, got {M3C2_FILTER!r}")

## M3C2 VISUALIZATION
M3C2_COLOR_BREAKS = (-2, -1, -0.3, 0.3, 1, 2)
EROSION_COLOR = "blue"

if EROSION_COLOR not in ("red", "blue"):
    raise ValueError(f"EROSION_COLOR must be 'red' or 'blue', got {EROSION_COLOR!r}")

print("Parameters set")

## 6. Initialize new run

In [ ]:
## Creates timestamped Outputs and Parameters folders
## saves run_config.json in the Parameters folder

CLOUDCOMPY_DIR, OUTPUTS_DIR, TILES_DIR, RASTERS_DIR, M3C2_DIR, VISUALIZATION_DIR, PARAMETERS_DIR, SCRATCH_DIR = define_project_directories(PROJECT_ROOT)
# TILES_DIR: retiled before/after point clouds (compressed) + raw-extent shapefiles
# RASTERS_DIR: before/after DEM, slope, hillshade tifs
# M3C2_DIR: M3C2 point clouds + merged M3C2 raster
# VISUALIZATION_DIR: jpg maps
# PARAMETERS_DIR: parameter files + run_config.json
# SCRATCH_DIR: this run's intermediates (tiles, DEM tiles, raster tiles);
#              deleted at the end of the change detection cell
# Preprocessed clouds are not here -- they stay in PREPROCESSED_DIR

RUN_RECORD = PARAMETERS_DIR / "run_config.json"

write_run_record(
    RUN_RECORD,
    globals(),
    extra={
        "aoi_crs": aoi_crs,
        "before_crs": before_crs,
        "after_crs": after_crs,
        "before_files": [f.name for f in before_files],
        "after_files": [f.name for f in after_files],
        "before_ground_points_in_aoi": before_points,
        "after_ground_points_in_aoi": after_points,
        "before_ground_density_pts_per_m2": round(before_density, 3),
        "after_ground_density_pts_per_m2": round(after_density, 3),
        "preprocess_config": preprocess_config,
    },
    cloudcompy_dir=CLOUDCOMPY_DIR,
    env=PDAL_ENV
)

## Copy the shared preprocessing's pipeline files and raw-extent shapefiles
## into this run's folders, so each Outputs folder stays a complete record
copied_artifacts = copy_preprocess_artifacts(PREPROCESSED_DIR, PARAMETERS_DIR, TILES_DIR)
print(f"Copied {len(copied_artifacts)} preprocessing file(s) from {PREPROCESSED_DIR}")

print(f"Outputs for this run: {OUTPUTS_DIR}")

## 7. Retile processed point clouds

In [ ]:
## This cell tiles the preprocessed point clouds using tiling parameters set in Cell 5
## Retains buffer to avoid edge effects in neighborhood processing
## Small tiles are then merged with adjacent larger tiles to minimize ICP errors later

working_tiles_dir = SCRATCH_DIR / "tiles"

print("Retiling point clouds")
retile_clouds(
    input_dir=PREPROCESSED_DIR/"before",
    output_dir=working_tiles_dir,
    input_pattern="*_processed.las",
    output_prefix="before_",
    tile_size=TILE_SIZE,
    env=PDAL_ENV
)

retile_clouds(
    input_dir=PREPROCESSED_DIR/"after",
    output_dir=working_tiles_dir,
    input_pattern="*_processed.las",
    output_prefix="after_",
    tile_size=TILE_SIZE,
    env=PDAL_ENV
)

## Flag partial tiles and merge each one into its largest neighbor
print("Merging partial tiles")
tile_merge_plan = merge_partial_tiles(
    tiles_dir=working_tiles_dir,
    before_prefix="before_",
    after_prefix="after_",
    tile_size=TILE_SIZE,
    min_size_fraction=PARTIAL_TILE_MIN_PERCENT / 100,
    pipeline_dir=SCRATCH_DIR/"merge_pipelines",
    env=PDAL_ENV
)

print("Partial-tile merging complete")

## Keep compressed copies of the final tiles with this run's outputs
archive_tiles(working_tiles_dir, TILES_DIR, ["before_", "after_"], env=PDAL_ENV)

## 8. Alignment, M3C2 change detection, and raster generation

In [ ]:
## Main change detection loop
## Returns tiled M3C2 point clouds and merged rasters
## Deletes this run's scratch folder; the shared Preprocessed cache is kept

## Warns if any parameter was changed after the 'Start a new run' cell saved them
warn_if_parameters_changed(RUN_RECORD, globals())

## Create M3C2 parameter files
surface_param_file, vertical_param_file = (
    create_m3c2_parameter_files(
        output_dir=PARAMETERS_DIR,
        normal_scale=NORMAL_SCALE,
        projection_scale=PROJECTION_SCALE,
        search_depth=SEARCH_DEPTH,
        registration_error=REGISTRATION_ERROR
    )
)

print("\nSurface parameter file:")
print(surface_param_file)
print("\nVertical parameter file:")
print(vertical_param_file)

process_tile_pairs(
    before_folder=working_tiles_dir,
    after_folder=working_tiles_dir,
    before_prefix="before_",
    after_prefix="after_",
    tile_size=TILE_SIZE,
    buffer_distance=BUFFER_DISTANCE,
    min_spacing=MIN_SPACING,
    vertical_param_file=vertical_param_file,
    surface_param_file=surface_param_file,
    m3c2_mode=M3C2_MODE,
    m3c2_filter=M3C2_FILTER,
    output_cloud_dir=M3C2_DIR,
    output_raster_dir=SCRATCH_DIR,
    merged_raster_path=M3C2_DIR/"m3c2_merged.tif",
    m3c2_raster_resolution=M3C2_RASTER_RESOLUTION,
    before_dem_resolution=BEFORE_DEM_RESOLUTION,
    after_dem_resolution=AFTER_DEM_RESOLUTION,
    dem_output_dir=SCRATCH_DIR/"dem_tiles",
    merged_before_dem_path=RASTERS_DIR/"before_dem.tif",
    merged_after_dem_path=RASTERS_DIR/"after_dem.tif",
    before_slope_path=RASTERS_DIR/"before_slope.tif",
    before_hillshade_path=RASTERS_DIR/"before_hillshade.tif",
    after_slope_path=RASTERS_DIR/"after_slope.tif",
    after_hillshade_path=RASTERS_DIR/"after_hillshade.tif",
    crop_polygon=crop_polygon,
    icp_min_rms_decrease=ICP_MIN_RMS_DECREASE,
    icp_sampling_limit=ICP_SAMPLING_LIMIT,
    output_crs=OUTPUT_CRS,
    tile_merge_plan=tile_merge_plan,
    env=PDAL_ENV
)

## Delete this run's scratch folder
## PREPROCESSED_DIR is left alone, so another run can reuse it
shutil.rmtree(SCRATCH_DIR, ignore_errors=True)
print(f"Removed scratch directory: {SCRATCH_DIR}")
print(f"Kept preprocessed point clouds: {PREPROCESSED_DIR}")

## 9. Generate visualizations of before/after topography and M3C2 change

In [ ]:
## Draws three maps, shows them below, and saves each as a jpg in VISUALIZATION_DIR:
##   1) BEFORE slopeshade
##   2) AFTER slopeshade
##   3) AFTER slopeshade with the M3C2 change overlay, colored by
##      M3C2_COLOR_BREAKS and EROSION_COLOR from the parameters cell

import matplotlib.pyplot as plt

from LidarChangeScripts.visualization import (
    read_raster_as_array,
    build_m3c2_color_bins,
    plot_slopeshade,
    change_map_drawer,
    save_maps_with_matching_crop
)

VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

## M3C2 change raster and its color scale
m3c2_change_label = M3C2_CHANGE_LABELS[(M3C2_MODE, M3C2_FILTER)]
m3c2, m3c2_extent = read_raster_as_array(M3C2_DIR / "m3c2_merged.tif")
m3c2_color_bins = build_m3c2_color_bins(M3C2_COLOR_BREAKS, erosion_color=EROSION_COLOR)

## draw_change_map(ax) draws the AFTER slopeshade + M3C2 change overlay on ax
## (also used by the swath profile cell)
draw_change_map = change_map_drawer(RASTERS_DIR, m3c2, m3c2_extent, m3c2_color_bins, m3c2_change_label)

## 1) BEFORE slopeshade
fig_before, ax_before = plt.subplots(figsize=(10, 10))
plot_slopeshade(
    ax_before,
    RASTERS_DIR / "before_hillshade.tif",
    RASTERS_DIR / "before_slope.tif",
    "BEFORE Slopeshade"
)

## 2) AFTER slopeshade
fig_after, ax_after = plt.subplots(figsize=(10, 10))
plot_slopeshade(
    ax_after,
    RASTERS_DIR / "after_hillshade.tif",
    RASTERS_DIR / "after_slope.tif",
    "AFTER Slopeshade"
)

## 3) AFTER slopeshade + M3C2 change overlay
fig_change, ax_change = plt.subplots(figsize=(10, 10))
draw_change_map(ax_change)

## Save all three with the same crop
save_maps_with_matching_crop(
    [
        (fig_before, VISUALIZATION_DIR / "before_slopeshade.jpg"),
        (fig_after, VISUALIZATION_DIR / "after_slopeshade.jpg"),
        (fig_change, VISUALIZATION_DIR / "after_slopeshade_m3c2.jpg"),
    ],
    reference=fig_change
)
plt.show()

## 10. Point cloud swath profile tool

In [ ]:
## Tool for generating point cloud swath profiles using interactive picker

## Profile line and swath properties
PROFILE_NAME = "A" # change name to make new profiles without overwriting earlier ones
PROFILE_START = None  # (easting, northing) in OUTPUT_CRS, or None to pick on the map
PROFILE_END = None # (easting, northing) in OUTPUT_CRS, or None to pick on the map
PROFILE_SWATH_WIDTH = 1.0  # swath width in meters

## CROSS-SLOPE TILT CORRECTION -- "none", "varying" or "constant"
## "varying" de-tilts along the profile by the local cross-slope. The swath collapses and
##   vertical change stays true, but the figure is a shear, not a view of the cloud
## "constant" rotates the cloud by one angle to preserve an orthographic view and the
##   swath collapses only where the ground matches it and vertical change reads low by cos(tilt)
PROFILE_TILT_MODE = "varying"
PROFILE_TILT_SMOOTHING = 3.0  # m, along-profile window the cross-slope is averaged over
PROFILE_TILT_ANGLE = None  # deg, "constant" mode only -- None takes the tilt from the DEM

## LEGEND LABELS (e.g. the survey dates)
PROFILE_BEFORE_LABEL = "Before"
PROFILE_AFTER_LABEL = "After"


import json
import math

from LidarChangeScripts.tile_grid import tile_paths
from LidarChangeScripts.swath_profile import (
    pick_profile_endpoints,
    load_icp_transforms,
    read_swath_points,
    cross_slope_along_profile,
    remove_cross_slope,
    describe_cross_slope,
    profile_tilt,
    tilt_swath_view,
    describe_tilt_view,
    plot_profile_location,
    plot_swath_section
)

if PROFILE_TILT_MODE not in ("none", "varying", "constant"):
    raise ValueError(
        f"PROFILE_TILT_MODE must be 'none', 'varying' or 'constant', got {PROFILE_TILT_MODE!r}"
    )

## Pick the profile line on the map, if it isn't set above
if PROFILE_START is None or PROFILE_END is None:
    PROFILE_START, PROFILE_END = pick_profile_endpoints(draw_change_map, PROFILE_SWATH_WIDTH, PROFILE_NAME)
    print("Profile endpoints -- to reproduce profile later, replace the None values above with:")
    print(f"PROFILE_START = {PROFILE_START}")
    print(f"PROFILE_END = {PROFILE_END}\n")

profile_length = math.dist(PROFILE_START, PROFILE_END)
print(f"Profile {PROFILE_NAME}: {profile_length:.1f} m long, {PROFILE_SWATH_WIDTH:g} m swath")

## Ground points in the swath: BEFORE tiles as saved, AFTER tiles ICP-aligned
icp_transforms = load_icp_transforms(M3C2_DIR / "icp_transforms.json")
after_label = PROFILE_AFTER_LABEL if icp_transforms else f"{PROFILE_AFTER_LABEL} (not ICP-aligned)"

before_along, before_across, before_z = read_swath_points(
    tile_paths(TILES_DIR, "before_"), PROFILE_START, PROFILE_END, PROFILE_SWATH_WIDTH,
    env=PDAL_ENV
)
after_along, after_across, after_z = read_swath_points(
    tile_paths(TILES_DIR, "after_"), PROFILE_START, PROFILE_END, PROFILE_SWATH_WIDTH,
    icp_transforms=icp_transforms, env=PDAL_ENV
)
print(f"Points in the swath: BEFORE {before_z.size:,}, AFTER {after_z.size:,}")

if before_z.size == 0 and after_z.size == 0:
    raise RuntimeError("No points in the swath -- is the profile inside the AOI?")

## Take the cross-swath slope out of the section (the same correction for both epochs,
## so every BEFORE-to-AFTER separation survives it untouched)
tilt_angle = None
tilt_note = None
## The N/E/Z triad is only drawn when one tilt covers the whole profile, so the
## section really is the cloud seen from one direction -- see the module docstring
profile_triad = (PROFILE_START, PROFILE_END, 0.0) if PROFILE_TILT_MODE == "none" else None

if PROFILE_TILT_MODE != "none":
    stations, cross_slope = cross_slope_along_profile(
        RASTERS_DIR / "before_dem.tif", PROFILE_START, PROFILE_END, smoothing=PROFILE_TILT_SMOOTHING
    )

if PROFILE_TILT_MODE == "varying":
    print(describe_cross_slope(cross_slope, PROFILE_SWATH_WIDTH))
    before_z = remove_cross_slope(before_along, before_across, before_z, stations, cross_slope)
    after_z = remove_cross_slope(after_along, after_across, after_z, stations, cross_slope)
    tilt_note = "cross-slope corrected"

elif PROFILE_TILT_MODE == "constant":
    tilt_angle = PROFILE_TILT_ANGLE if PROFILE_TILT_ANGLE is not None else profile_tilt(cross_slope)
    print(describe_tilt_view(cross_slope, tilt_angle, PROFILE_SWATH_WIDTH))
    before_z = tilt_swath_view(before_across, before_z, tilt_angle)
    after_z = tilt_swath_view(after_across, after_z, tilt_angle)
    tilt_note = f"tilted {tilt_angle:.0f} deg"
    profile_triad = (PROFILE_START, PROFILE_END, tilt_angle)

## Generate location map
fig_map, ax_map = plt.subplots(figsize=(8, 8))
draw_change_map(ax_map)
plot_profile_location(ax_map, PROFILE_START, PROFILE_END, PROFILE_SWATH_WIDTH, PROFILE_NAME)

## Plot cross section
fig_section, ax_section = plt.subplots(figsize=(12, 6))
plot_swath_section(
    ax_section,
    [
        (before_along, before_z, PROFILE_BEFORE_LABEL, "black"),
        (after_along, after_z, after_label, "red"),
    ],
    length=profile_length,
    name=PROFILE_NAME,
    width=PROFILE_SWATH_WIDTH,
    tilt_note=tilt_note,
    triad=profile_triad
)

fig_map.savefig(VISUALIZATION_DIR / f"profile_{PROFILE_NAME}_map.jpg", dpi=200, facecolor="white", bbox_inches="tight")
fig_section.savefig(VISUALIZATION_DIR / f"profile_{PROFILE_NAME}_section.jpg", dpi=200, facecolor="white", bbox_inches="tight")

## This profile's settings, saved next to its figures so it can be re-made
with open(VISUALIZATION_DIR / f"profile_{PROFILE_NAME}.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "start": PROFILE_START,
            "end": PROFILE_END,
            "length_m": round(profile_length, 2),
            "swath_width_m": PROFILE_SWATH_WIDTH,
            "tilt_mode": PROFILE_TILT_MODE,
            "tilt_smoothing_m": PROFILE_TILT_SMOOTHING,
            "tilt_angle_deg": round(tilt_angle, 2) if tilt_angle is not None else None,
            "after_icp_aligned": icp_transforms is not None,
            "before_points": int(before_z.size),
            "after_points": int(after_z.size),
        },
        f,
        indent=2
    )

print(f"Saved profile_{PROFILE_NAME}_map.jpg, profile_{PROFILE_NAME}_section.jpg and profile_{PROFILE_NAME}.json to {VISUALIZATION_DIR}")
plt.show()